# Energy Load Forecasting

## Project Overview

Forecasts **daily energy load** (GWh) over a 14-day horizon. Synthetic data spans 2 years with weekly cycles, seasonal variation, and economic growth effects.

**Models**: Naive, LazyPredict, FLAML, StatsForecast. **Horizon**: 14 days.

## Learning Objectives

1. Understand time-series patterns (trend, seasonality, noise)
2. Build naive and seasonal-naive baselines
3. Engineer lag and rolling features for tabular ML
4. Use LazyPredict for quick regression benchmarking
5. Apply FLAML AutoML (excluding XGBoost due to compatibility)
6. Use StatsForecast (AutoARIMA, AutoETS, SeasonalNaive)
7. Compare all approaches with MAE / RMSE / MAPE

## Problem Statement

Given historical daily energy consumption, predict the next 14 days. Energy load forecasting drives generation planning, market bidding, and infrastructure investment decisions.

## Why This Project Matters

Energy load forecasting underpins electricity market operations. Utilities bid generation capacity based on load forecasts. Errors translate directly to financial losses or reliability risks.

## Dataset Overview

Synthetic dataset:
- 730 days (2 years) of daily energy load
- Weekly cycle (weekday industrial load, lower weekends)
- Seasonal variation (summer/winter peaks)
- Economic growth trend
- Holiday reductions

| Column | Description |
|--------|-------------|
| `date` | Date |
| `load_gwh` | Daily total energy load (GWh) |

## Dataset Source and License Notes

Synthetically generated for educational purposes. No external download required.

## Environment Setup

In [1]:
import subprocess, importlib
for pkg in ['numpy', 'pandas', 'matplotlib', 'scikit-learn', 'statsforecast', 'flaml', 'lazypredict']:
    try:
        importlib.import_module(pkg.replace('-', '_').split('[')[0])
    except ImportError:
        subprocess.check_call(['pip', 'install', '-q', pkg])
print("All packages ready.")

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## Imports

In [2]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from sklearn.metrics import mean_absolute_error, mean_squared_error
print("Imports complete.")

Imports complete.


## Configuration / Constants

In [3]:
SEED = 42
N_POINTS = 730
HORIZON = 14
VAL_SIZE = 14
TEST_SIZE = 14
SEASON_LENGTH = 7
FREQ = 'D'
TARGET = 'load_gwh'
np.random.seed(SEED)
print(f"Config: {N_POINTS} points, horizon={HORIZON}, season={SEASON_LENGTH}")

Config: 730 points, horizon=14, season=7


## Dataset Generation

In [4]:
dates = pd.date_range(start='2022-01-01', periods=N_POINTS, freq='D')
t = np.arange(N_POINTS)

base = 80 + 0.01 * t  # GWh scale
weekly = np.zeros(N_POINTS)
for i in range(N_POINTS):
    dow = dates[i].dayofweek
    if dow <= 4: weekly[i] = 5
    else: weekly[i] = -8

seasonal = 10 * np.sin(2 * np.pi * (t - 45) / 365)  # winter peak

holiday = np.zeros(N_POINTS)
for i in range(N_POINTS):
    m, d = dates[i].month, dates[i].day
    if (m == 12 and d >= 24) or (m == 1 and d <= 2): holiday[i] = -15
    elif m == 11 and 22 <= d <= 28: holiday[i] = -10

noise = np.random.normal(0, 3, N_POINTS)
load_gwh = base + weekly + seasonal + holiday + noise
load_gwh = np.maximum(load_gwh, 30).round(1)

df = pd.DataFrame({'date': dates, 'load_gwh': load_gwh})
print(f"Dataset shape: {df.shape}")
df.head()

Dataset shape: (730, 2)


,date,load_gwh
0,2022-01-01,51.5
1,2022-01-02,49.7
2,2022-01-03,80.2
3,2022-01-04,83.0
4,2022-01-05,77.9


## Data Validation Checks

In [5]:
assert df.isnull().sum().sum() == 0, "Missing values"
assert (df[TARGET] > 0).all(), "Non-positive values"
assert df['date'].is_monotonic_increasing, "Not sorted"
assert len(df) == N_POINTS, "Row count mismatch"
print("All validation checks passed.")

All validation checks passed.


## Exploratory Data Analysis

In [6]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes[0, 0].plot(df['date'], df[TARGET], linewidth=0.6)
axes[0, 0].set_title('load_gwh Over Time')
axes[0, 1].hist(df[TARGET], bins=30, edgecolor='black', alpha=0.7)
axes[0, 1].set_title('Distribution')
from pandas.plotting import autocorrelation_plot
autocorrelation_plot(df[TARGET], ax=axes[1, 0])
axes[1, 0].set_xlim(0, min(60, N_POINTS // 2))
axes[1, 0].set_title('Autocorrelation')
rw = min(SEASON_LENGTH * 2, N_POINTS // 4)
axes[1, 1].plot(df['date'], df[TARGET].rolling(rw).mean())
axes[1, 1].set_title(f'Rolling {rw}-period Mean')
plt.tight_layout()
plt.savefig('eda.png', dpi=100, bbox_inches='tight')
plt.show()
print("EDA complete.")

EDA complete.


## Target Analysis

In [7]:
print("load_gwh Statistics:")
print(df[TARGET].describe())
print(f"\nCV: {df[TARGET].std() / df[TARGET].mean():.3f}")
print(f"Skewness: {df[TARGET].skew():.3f}")

load_gwh Statistics:
count    730.000000
mean      84.261644
std       10.259754
min       49.700000
25%       78.200000
50%       84.350000
75%       92.600000
max      108.200000
Name: load_gwh, dtype: float64

CV: 0.122
Skewness: -0.439


## Train / Validation / Test Split Strategy

Time-aware split (no shuffling):
- **Train**: all except last 28
- **Validation**: 14 points
- **Test**: last 14 points

In [8]:
train = df.iloc[:-(VAL_SIZE + TEST_SIZE)].copy()
val = df.iloc[-(VAL_SIZE + TEST_SIZE):-TEST_SIZE].copy()
test = df.iloc[-TEST_SIZE:].copy()
print(f"Train: {len(train)}, Val: {len(val)}, Test: {len(test)}")

Train: 702, Val: 14, Test: 14


## Preprocessing Strategy

Minimal preprocessing — tree models handle raw features. Features created next.

In [9]:
df_full = df.copy().sort_values('date').reset_index(drop=True)
print('Data ready.')

Data ready.


## Feature Engineering

In [10]:
def create_features(data):
    d = data.copy()
    d['dow'] = d['date'].dt.dayofweek
    d['month'] = d['date'].dt.month
    d['day'] = d['date'].dt.day
    d['week_of_year'] = d['date'].dt.isocalendar().week.astype(int)
    for lag in [1, 7, 14, 28]:
        d[f'lag_{lag}'] = d[TARGET].shift(lag)
    for w in [7, 14, 28]:
        d[f'rmean_{w}'] = d[TARGET].shift(1).rolling(w).mean()
        d[f'rstd_{w}'] = d[TARGET].shift(1).rolling(w).std()
    return d

df_feat = create_features(df_full).dropna().reset_index(drop=True)
feature_cols = [c for c in df_feat.columns if c not in ['date', TARGET]]
print(f"Features: {len(feature_cols)} columns, {len(df_feat)} rows")

Features: 14 columns, 702 rows


## Baseline Model — Naive Forecasts

In [11]:
def calc_metrics(y_true, y_pred, name=""):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / np.maximum(y_true, 1))) * 100
    print(f"{name:30s} | MAE: {mae:10.1f} | RMSE: {rmse:10.1f} | MAPE: {mape:5.2f}%")
    return {'model': name, 'MAE': mae, 'RMSE': rmse, 'MAPE': mape}

results = []
naive_pred = np.full(TEST_SIZE, train[TARGET].iloc[-1])
results.append(calc_metrics(test[TARGET].values, naive_pred, "Naive (Last Value)"))

src = df_full[TARGET].values
ti = len(df_full) - TEST_SIZE
sn_pred = src[ti - SEASON_LENGTH:ti - SEASON_LENGTH + TEST_SIZE]
results.append(calc_metrics(test[TARGET].values, sn_pred, f"Seasonal Naive ({SEASON_LENGTH})"))

Naive (Last Value)             | MAE:        9.3 | RMSE:       11.2 | MAPE: 14.00%
Seasonal Naive (7)             | MAE:        8.2 | RMSE:       10.5 | MAPE: 12.66%


## LazyPredict Benchmark (Lag-Feature Tabular)

In [12]:
from lazypredict.Supervised import LazyRegressor

ct = len(df_feat) - TEST_SIZE
cv = ct - VAL_SIZE
X_tr = df_feat.iloc[:cv][feature_cols]
y_tr = df_feat.iloc[:cv][TARGET]
X_va = df_feat.iloc[cv:ct][feature_cols]
y_va = df_feat.iloc[cv:ct][TARGET]

reg = LazyRegressor(verbose=0, ignore_warnings=True, custom_metric=None, predictions=True)
lp_models, lp_preds = reg.fit(X_tr, X_va, y_tr, y_va)
print("\nLazyPredict Top 10:")
print(lp_models.head(10).to_string())


LazyPredict Top 10:
                            Adjusted R-Squared   R-Squared       RMSE  Time Taken
Model                                                                            
KernelRidge                        2641.755968 -202.135074  88.594725    0.065862
Lars                               1009.905361  -76.608105  54.760694    0.018311
GaussianProcessRegressor            765.420134  -57.801549  47.666118    0.234611
MLPRegressor                         75.594342   -4.738026  14.890068    0.634407
PassiveAggressiveRegressor           29.990848   -1.230065   9.282697    0.017742
QuantileRegressor                    27.082326   -1.006333   8.804747    0.134731
DummyRegressor                       26.815386   -0.985799   8.759575    0.011573
ExtraTreeRegressor                   20.494516   -0.499578   7.612021    0.017248
RANSACRegressor                      19.770751   -0.443904   7.469380    0.107191
DecisionTreeRegressor                19.492874   -0.422529   7.413887    0.02

## FLAML AutoML Run

In [13]:
from flaml import AutoML

automl = AutoML()
automl.fit(
    X_train=X_tr, y_train=y_tr,
    task='regression', time_budget=30, metric='mae',
    estimator_list=['lgbm', 'rf', 'extra_tree', 'catboost'],
    seed=SEED, verbose=0
)
print(f"FLAML best: {automl.best_estimator}")
flaml_val = automl.predict(X_va)
results.append(calc_metrics(y_va.values, flaml_val, f"FLAML ({automl.best_estimator})"))

X_te = df_feat.iloc[ct:][feature_cols]
y_te = df_feat.iloc[ct:][TARGET]
flaml_test = automl.predict(X_te)
results.append(calc_metrics(y_te.values, flaml_test, f"FLAML Test ({automl.best_estimator})"))

FLAML best: catboost
FLAML (catboost)               | MAE:        3.6 | RMSE:        4.5 | MAPE:  4.60%
FLAML Test (catboost)          | MAE:        6.1 | RMSE:        7.0 | MAPE:  9.44%


## StatsForecast — Statistical Forecasting

In [14]:
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, AutoETS, SeasonalNaive as SFSeasonalNaive

sf_df = df_full[['date', TARGET]].copy()
sf_df.columns = ['ds', 'y']
sf_df['unique_id'] = 'series1'
sf_df = sf_df[['unique_id', 'ds', 'y']]

sf_train = sf_df.iloc[:-TEST_SIZE]
sf = StatsForecast(
    models=[AutoARIMA(season_length=SEASON_LENGTH), AutoETS(season_length=SEASON_LENGTH),
            SFSeasonalNaive(season_length=SEASON_LENGTH)],
    freq=FREQ, n_jobs=1
)
sf.fit(sf_train)
sf_preds = sf.predict(h=TEST_SIZE).reset_index()

y_test_sf = test[TARGET].values
for col in ['AutoARIMA', 'AutoETS', 'SeasonalNaive']:
    if col in sf_preds.columns:
        results.append(calc_metrics(y_test_sf, sf_preds[col].values, f"SF {col}"))
print("StatsForecast complete.")

SF AutoARIMA                   | MAE:        9.2 | RMSE:       11.5 | MAPE: 14.65%
SF AutoETS                     | MAE:        9.3 | RMSE:       11.7 | MAPE: 14.76%
SF SeasonalNaive               | MAE:        9.8 | RMSE:       12.1 | MAPE: 15.48%
StatsForecast complete.


## Top 3 Model Selection

In [15]:
results_df = pd.DataFrame(results).sort_values('MAE').reset_index(drop=True)
print("All Results:")
print(results_df.to_string(index=False))
print("\nTop 3:")
print(results_df.head(3).to_string(index=False))

All Results:
                model      MAE      RMSE      MAPE
     FLAML (catboost) 3.637924  4.495191  4.595270
FLAML Test (catboost) 6.075521  6.999713  9.436573
   Seasonal Naive (7) 8.178571 10.548019 12.660281
         SF AutoARIMA 9.213673 11.466819 14.651886
           SF AutoETS 9.272859 11.650387 14.757052
   Naive (Last Value) 9.278571 11.162725 13.996347
     SF SeasonalNaive 9.750000 12.146399 15.480866

Top 3:
                model      MAE      RMSE      MAPE
     FLAML (catboost) 3.637924  4.495191  4.595270
FLAML Test (catboost) 6.075521  6.999713  9.436573
   Seasonal Naive (7) 8.178571 10.548019 12.660281


## Final Training and Evaluation of Top 3

In [16]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(test['date'].values, test[TARGET].values, 'ko-', label='Actual', ms=4)
ax.plot(test['date'].values, flaml_test, 's--', label=f'FLAML ({automl.best_estimator})', ms=4)
for col in ['AutoARIMA', 'AutoETS']:
    if col in sf_preds.columns:
        ax.plot(test['date'].values[:len(sf_preds)], sf_preds[col].values, 'o--', label=f'SF {col}', ms=4)
ax.set_title('Forecast vs Actual — Test Set')
ax.legend()
plt.tight_layout()
plt.savefig('forecast_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

## Error Analysis

In [17]:
errors = test[TARGET].values - flaml_test
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(range(len(errors)), errors, alpha=0.7)
axes[0].set_title('Residuals (Actual - FLAML)')
axes[0].axhline(y=0, color='r', linestyle='--')
axes[1].hist(errors, bins=10, edgecolor='black', alpha=0.7)
axes[1].set_title('Residual Distribution')
plt.tight_layout()
plt.savefig('error_analysis.png', dpi=100, bbox_inches='tight')
plt.show()
print(f"Mean residual: {np.mean(errors):.2f}, Std: {np.std(errors):.2f}")

Mean residual: -3.23, Std: 6.21


## Interpretation and Business Insight

- Energy load has strong weekday/weekend and seasonal patterns
- Industrial load drives weekday peaks
- Winter heating creates the dominant seasonal peak
- Holiday reductions are 15-20% below normal
- Economic growth adds a slow upward trend

## Limitations

1. Synthetic — real load depends on weather, economy, industrial activity
2. No weather features
3. No sector decomposition
4. Daily aggregation misses intraday peaks

## How to Improve This Project

1. Add weather data (temperature, humidity)
2. Include economic indicators (GDP growth, industrial production)
3. Segment by voltage level (transmission vs distribution)
4. Use hourly data for market bidding
5. Add renewable penetration effects

## Production Considerations

- Day-ahead and medium-term forecasting
- Integration with energy market platforms
- Weather-adjusted forecast updates
- Capacity planning dashboards

## Common Mistakes

1. Ignoring weather-load correlation
2. Not differentiating industrial vs residential patterns
3. Using annual averages for daily planning
4. Not accounting for efficiency improvements over time

## Mini Challenge / Exercises

1. Add synthetic temperature and measure load-temperature correlation
2. Decompose into trend + seasonal + residual components
3. Try monthly aggregation for long-term planning
4. Build a peak-load probability forecast

## Final Summary / Key Takeaways

1. Energy load forecasting is foundational for utility operations
2. Weekly and seasonal patterns are highly predictable
3. Weather is the key missing driver
4. ML models capture complex non-linear patterns
5. Combining FLAML with AutoARIMA gives robust performance

In [18]:
import json
metrics = {
    'project': 'Energy Load Forecasting',
    'best_model': results_df.iloc[0]['model'],
    'best_MAE': float(results_df.iloc[0]['MAE']),
    'best_RMSE': float(results_df.iloc[0]['RMSE']),
    'best_MAPE': float(results_df.iloc[0]['MAPE']),
    'all_results': results_df.to_dict('records')
}
with open('metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print("Metrics saved.")
print("\n=== Energy Load Forecasting — Complete ===")

Metrics saved.

=== Energy Load Forecasting — Complete ===
